<!-- DOC-BANNER -->
# One-Hot Encoding (Day 27)

> **Playlist:** CampusX Day 27 *One Hot Encoding | Handling Categorical Data*.
> **Purpose:** One-hot on `cars.csv` (`brand/km_driven/fuel/owner/selling_price`, ~8128 rows, train 5689): `value_counts(owner)` → `get_dummies(fuel, owner)` full + `drop_first` (k-1, dummy-trap) → `train_test_split(.3, rs=34)` → `OneHotEncoder(drop='first')` fit/transform train (5689,7) → `hstack` (5689,9) → top-category `brand` threshold-100 → `replace` rare→other → uncommon `get_dummies` sample.
> **After studying:** you can explain the dummy trap / multicollinearity-zero argument and handle high-cardinality with top-k grouping.


## One-hot encoding: equal-status categories get their own columns

Use it when **no order exists** between levels (fuel type, owner type, brand). Beware the dummy-variable trap: for n levels, keep n-1 columns.


### Dummy-variable trap: for n levels, keep n-1

Input columns must not predict each other (multicollinearity must be zero). n-1 columns fully represent n levels — the dropped level is the all-zero baseline.


In [1]:
import matplotlib.pyplot as plt 
import  seaborn as sns 
import pandas as pd 
import numpy as  np 



### 2. High cardinality — top-category grouping
Brands with <100 rows → `other` via `replace`, then dummies. Keeps width manageable without dropping the whole column.

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(
    style="darkgrid",
    context="notebook",
    palette="husl",
    font="sans-serif",
    font_scale=1.1,
    rc={
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
        "lines.linewidth": 2.5,
        "lines.markersize": 7,
        "legend.frameon": False,
        "figure.autolayout": True
    }
)

plt.rcParams["figure.facecolor"] = "#1e1e2f"
plt.rcParams["axes.facecolor"] = "#282840"
plt.rcParams["axes.labelcolor"] = "#eeeeff"
plt.rcParams["text.color"] = "#eeeeff"
plt.rcParams["xtick.color"] = "#ccccdd"
plt.rcParams["ytick.color"] = "#ccccdd"
plt.rcParams["grid.color"] = "#777799"

In [3]:
cars = pd.read_csv(r"E:\MACHINE_LEARNING\DATA_SETS\cars.csv")

In [4]:
cars

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000
...,...,...,...,...,...
8123,Hyundai,110000,Petrol,First Owner,320000
8124,Hyundai,119000,Diesel,Fourth & Above Owner,135000
8125,Maruti,120000,Diesel,First Owner,382000
8126,Tata,25000,Diesel,First Owner,290000


### Many levels? Group the rare ones as `other`

Keep frequent categories, merge the rest into one `other` bucket — this bounds the width without dropping the column.


In [5]:
df = cars
df['owner'].value_counts()


owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

### One-hot with pandas: `get_dummies`

Quick and handy for exploration; for production pipelines prefer sklearn (it remembers the categories).


### 1. `get_dummies` vs `OneHotEncoder`
`pd.get_dummies` = quick EDA; `OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')` = pipeline-safe (remembers categories, ignores unseen). `drop_first` keeps n-1 dummies to avoid the dummy trap.

In [6]:
pd.get_dummies(df , columns=['fuel' , 'owner'])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


### k-1 encoding: drop one level

`drop_first=True` keeps n-1 dummy columns and avoids the dummy trap.


In [7]:
pd.get_dummies(df , columns= ['fuel' , 'owner'] , drop_first =True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


### One-hot with sklearn: `OneHotEncoder`

Pipeline-safe: remembers train categories and handles unseen test levels with `handle_unknown="ignore"`.


In [8]:

from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(df.iloc[: , 0 : 4 ],df.iloc[: , :-1]  ,test_size= .3 ,  random_state = 34)
df.sample()


,brand,km_driven,fuel,owner,selling_price
631,Hyundai,50000,Petrol,First Owner,235000


In [9]:
from sklearn.preprocessing import OneHotEncoder


In [10]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='first', sparse_output=False,dtype= np.int32)

In [11]:
X_train_new = ohe.fit_transform(X_train[['fuel' , 'owner']])
X_train_new

array([[0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 0, 0]], shape=(5689, 7), dtype=int32)

In [12]:
X_test_new = ohe.fit_transform(X_test[['fuel' , 'owner']])

In [13]:
np.hstack((X_train[['brand' ,'km_driven']].values , X_train_new )).shape

(5689, 9)

### One-hot with top categories

Group rare brands into `other` first, then encode — same grouping idea applied end to end.


In [14]:
counts = df['brand'].value_counts()



In [15]:
df['brand'].nunique()
threshold = 100 

In [16]:
repl = counts[counts <= threshold ].index

In [17]:
pd.get_dummies(df['brand'].replace(repl ,'unommon')).sample(5)


,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,unommon
4522,False,False,False,False,True,False,False,False,False,False,False,False,False
1756,False,False,False,False,True,False,False,False,False,False,False,False,False
5473,False,False,False,True,False,False,False,False,False,False,False,False,False
5755,False,False,False,False,False,False,True,False,False,False,False,False,False
5459,False,False,False,False,True,False,False,False,False,False,False,False,False


<!-- EXPERIMENT -->
### Experiment: move the rarity threshold
Change the top-category cutoff from `100` to `500` and predict: more or fewer brands collapse into `other`, and does the encoded width grow or shrink? Then fix the documented `y` overlap yourself — select only the target column and confirm X/y no longer share one.

<!-- DOC-BANNER -->
## Key takeaways (Day 27)
- Unordered inputs → OneHot; drop one level (n-1) when multicollinearity matters.
- **Bugs preserved (fix before re-run):** `y=df.iloc[:,:-1]` overlaps X (should be the target column); `fit_transform` on `X_test` leaks (use `transform`); typo `unommon`; double `OneHotEncoder` import. No model trained here — modeling starts Day 29.
